# **Scraping Planos Ultranet**

_Página do provedor: https://ultra.psi.br/home/_

In [ ]:
# Bibliotecas
import pandas as pd
from bs4 import BeautifulSoup
from datetime import date

## **1. Extração**

In [ ]:
# Obter lista de planos
texto_html = open("../../data/scraped/ultranet.html", encoding="utf-8")
soup = BeautifulSoup(texto_html, "html.parser")
container_planos = soup.find("section")
planos = container_planos.find_all("div", class_="elementor-widget-wrap")
print(f"{len(planos)} cartões de planos capturados.")

In [ ]:
# Extrair informações dos planos
def raspar_plano(plano):
    velocidade = plano.find_all("h3", class_="elementor-heading-title")[0].text
    preco = plano.find_all("h3", class_="elementor-heading-title")[1].text
    svas = ", ".join(span.get_text(strip=True) for span in plano.find_all("span", class_="elementor-icon-list-text"))

    return {
        "velocidade": velocidade or None,
        "preco": preco or None,
        "svas": svas
    }

df = pd.DataFrame(raspar_plano(p) for p in planos).drop_duplicates().reset_index(drop=True)
df

## **2. Tratamento**

In [ ]:
df_tratado = df.copy()

# Data e provedor
df_tratado["DATA"] = date.today().strftime("%Y-%m-%d")
df_tratado["ISP"] = "Ultranet"

# Velocidade
df_tratado["VELOCIDADE"] = df_tratado["velocidade"].str.extract(r"(\d+)")

# SVAs
df_tratado["SVAs"] = df_tratado["svas"]

# PRECO
df_tratado["PRECO"] = (
    df_tratado["preco"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .str.extract(r"(\d+\.?\d*)")[0]
    .astype(float)
)

#Seleção de colunas
df_tratado = df_tratado[["DATA", "ISP", "VELOCIDADE", "SVAs", "PRECO"]]

df_tratado

## **3. Carga**

In [ ]:
# Salvando
provedor = "ULTRANET.csv"
nome_arq = df_tratado["DATA"].max() + "-" + provedor

# Na rede Valenet
pasta = r"\\fileserver\arquivos\Estrategia Comercial\2. POWER BI\ECM - Radar Mercado\ScrapingPlanos"
df_tratado.to_csv(fr"{pasta}\{nome_arq}", index=False)

# Neste Workspace
df_tratado.to_csv(f"../../data/flat/{provedor}", index=False)

print(f"🟢 {nome_arq} salvo!")